<a href="https://colab.research.google.com/github/afullhart/climateanalogs/blob/main/Colab/Accuracy_Score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import rioxarray as rx
import geopandas as gpd
from rasterio.features import rasterize

# 1. Define Paths
iso_raster_path = '/content/drive/My Drive/Colab Notebooks/Analogs/IsoCluster.tif'
eco3_shp_path = '/content/drive/My Drive/Colab Notebooks/Analogs/us_eco_l3_clip/us_eco_l3_clip.shp'
zonalf_path = '/content/drive/My Drive/Colab Notebooks/Analogs/zonal_stats.csv'

# 2. Load the IsoCluster array
iso_im = rx.open_rasterio(iso_raster_path)
iso_arr = iso_im.values[0, :, :]

# 3. Map Original Cluster ID (1-15) to Temperature Order (1-15) natively
raw_df = pd.read_csv(zonalf_path)
sorted_indices = raw_df['MEAN'].sort_values(ascending=False).index
temp_to_orig_map = {rank + 1: original_idx + 1 for rank, original_idx in enumerate(sorted_indices)}

# 4. Load and Rasterize the Eco3 Shapefile
# Note: Ensure 'US_L3CODE' is cast to integer for the array matching
elc_gdf = gpd.read_file(eco3_shp_path)
elc_arr = rasterize(
    shapes=((geom, int(value)) for geom, value in zip(elc_gdf.geometry, elc_gdf['US_L3CODE'])),
    out_shape=iso_arr.shape,
    transform=iso_im.rio.transform(),
    fill=0,
    dtype='int16'
)

# 5. Mask and flatten to only evaluate valid overlapping pixels
valid_mask = (iso_arr >= 1) & (iso_arr <= 15) & (elc_arr > 0)
iso_flat = iso_arr[valid_mask]
elc_flat = elc_arr[valid_mask]

# 6. Generate the master Spatial Overlap Matrix (In-Memory Zonal Histogram)
overlap_matrix = pd.crosstab(iso_flat, elc_flat)
total_valid_pixels = len(iso_flat)

# 7. Define your Case-Normalized Groups (Temperature Ordered)
# Example: Using the ZeroOne optimized groups from our previous chat
cn_groups = {
    'Hot Desert': {'target_ecocodes': [14, 81], 'temp_clusters': [1, 2, 3]},
    'Great Basin/Colorado Plateau': {'target_ecocodes': [13, 20, 22, 24], 'temp_clusters': [6, 7, 8, 9, 10, 11, 12, 14]},
    'Northern Basin/Mountain': {'target_ecocodes': [5, 18, 19, 21, 80], 'temp_clusters': [12, 13, 14, 15]},
    'Mogollon/Madrean': {'target_ecocodes': [23, 79], 'temp_clusters': [3, 5, 13]},
    'Great Plains': {'target_ecocodes': [25, 26], 'temp_clusters': [4, 11]}
}

# 8. Calculate Accuracy Metrics Dynamically
results = []
for group_name, params in cn_groups.items():

    # Map target ecocodes and original cluster IDs
    target_ecos = [eco for eco in params['target_ecocodes'] if eco in overlap_matrix.columns]
    orig_clusters = [temp_to_orig_map[t] for t in params['temp_clusters']]

    # Calculate Base Areas
    elc_area = overlap_matrix[target_ecos].sum().sum()
    outside_area = total_valid_pixels - elc_area
    cluster_area = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters)].sum().sum()

    # Calculate Confusion Matrix
    true_positives = overlap_matrix.loc[overlap_matrix.index.isin(orig_clusters), target_ecos].sum().sum()
    false_positives = cluster_area - true_positives
    false_negatives = elc_area - true_positives
    true_negatives = outside_area - false_positives

    # Calculate Rates
    tpr = true_positives / elc_area if elc_area > 0 else 0
    tnr = true_negatives / outside_area if outside_area > 0 else 0
    case_normalized_rate = (tpr + tnr) / 2
    accuracy = (true_positives + true_negatives) / total_valid_pixels

    results.append({
        'Diagnostic Group': group_name,
        'TP': true_positives,
        'FP': false_positives,
        'TN': true_negatives,
        'FN': false_negatives,
        'Accuracy': round(accuracy, 4),
        'Case-Normalized Rate': round(case_normalized_rate, 4)
    })

# 9. Output the final metrics
metrics_df = pd.DataFrame(results)
print(metrics_df.to_string(index=False))